# 🏺 Hieroglyph NLP Pipeline — Kaggle + Seed-X-PPO-7B + Ngrok
### Transliteration → Dictionary → spaCy → Seed-X Arabic Translation → Sentiment
---
> **GPU: Tesla T4 (16GB)** — Seed-X-PPO-7B بيشتغل على الـ T4 كويس جداً

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'⚠️  {result.stderr[-300:]}')

# numpy الأول عشان يمنع binary incompatibility
pip('numpy==1.26.4')

# Core deps
pip('protobuf==3.20.3')
pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
pip('flask', 'flask-cors', 'pyngrok')

# spaCy
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')
print('⚠️  Restart the kernel now (Run → Restart & Run All)')

## Cell 1 — Dataset Paths
> تأكد إن الـ dataset اسمه `hieroglyph-data` في الـ Input

In [1]:
import os

# ── حاول الـ path الجديد الأول، لو مش موجود جرب القديم ──
def find_path(filename, candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

GARDINER_PATH = "/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv"
DICT_PATH = "/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv"
INTENTION_PATH = "/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv"

for label, path in [('Gardiner', GARDINER_PATH), ('Dictionary', DICT_PATH), ('Intention', INTENTION_PATH)]:
    status = '✅' if path else '❌ NOT FOUND'
    print(f'  {status}  {label}: {path}')

  ✅  Gardiner: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv
  ✅  Dictionary: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv
  ✅  Intention: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv


## Cell 2 — Load Gardiner Sign List

In [2]:
import pandas as pd

df = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
for _, row in df.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic': str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning' : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode' : str(row['unicode']).strip()  if pd.notna(row['unicode'])  else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')

✅ Gardiner Sign List loaded: 819 signs
   Signs with phonetic: 224


## Cell 3 — Load Egyptian Dictionary

In [3]:
from collections import defaultdict

df_dict = pd.read_csv(DICT_PATH)
_raw = defaultdict(list)
for _, row in df_dict.iterrows():
    key = str(row['transliteration']).strip()
    val = str(row['english']).strip()
    if not key or key == 'nan' or not val or val == 'nan':
        continue
    if val not in _raw[key]:
        _raw[key].append(val)

EGYPTIAN_DICT = dict(_raw)
print(f'✅ Egyptian Dictionary loaded: {len(EGYPTIAN_DICT)} entries')

✅ Egyptian Dictionary loaded: 1352 entries


## Cell 4 — Load Intention Dataset

In [4]:
df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip()
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }

print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')

✅ Intention dataset loaded: 139 intentions


## Cell 5 — Load spaCy + Sentiment Models

In [5]:
import torch
import numpy as np
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── spaCy ────────────────────────────────────────────────────
nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')

# ── Sentiment (CPU — نسيب الـ VRAM كلها لـ Seed-X) ──────────
print('🔄 Loading sentiment model on CPU...')
SENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer  = AutoTokenizer.from_pretrained(SENT_MODEL_NAME)
sent_model      = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
sent_model      = sent_model.to('cpu').eval()
print('✅ Sentiment model loaded (CPU)')

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB
✅ spaCy loaded
🔄 Loading sentiment model on CPU...


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

✅ Sentiment model loaded (CPU)


## Cell 6 — Load Seed-X-PPO-7B (vllm)
> ⏳ أول مرة: بياخد 3-5 دقايق للتحميل من HuggingFace
> من الـ cache بيبقى في ثواني

In [6]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
from safetensors import safe_open
import transformers.modeling_utils as _mu
import torch

# ── Monkey-patch: إصلاح safetensors metadata bug على Kaggle ──
_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass
_mu.safe_open = _PatchedSafeOpen

SEED_X_MODEL = 'ByteDance-Seed/Seed-X-PPO-7B'
print(f'🔄 Loading {SEED_X_MODEL} ...')
print('   float16 — T4 لا تدعم bfloat16')
print('   أول مرة: ~2 دقيقة — من الـ cache: ثواني')

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    SEED_X_MODEL,
    trust_remote_code=True,
)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    SEED_X_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
seed_x_model.eval()
print(f'✅ {SEED_X_MODEL} loaded and ready!')

🔄 Loading ByteDance-Seed/Seed-X-PPO-7B ...
   float16 — T4 لا تدعم bfloat16
   أول مرة: ~2 دقيقة — من الـ cache: ثواني


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ ByteDance-Seed/Seed-X-PPO-7B loaded and ready!


## Cell 7 — NLP Functions

In [7]:
import re

# ─────────────────────────────────────────────────────────────
# TRANSLITERATION
# ─────────────────────────────────────────────────────────────

def extract_core_meaning(meanings):
    text  = meanings[0] if isinstance(meanings, list) else str(meanings)
    first = re.split(r'[/|]', text)[0].strip()
    return re.sub(r'^be ', '', first).strip().lower()


def build_sentence_spacy(core_meanings):
    if not core_meanings:
        return ''
    if len(core_meanings) == 1:
        return core_meanings[0]

    doc   = nlp_spacy(' '.join(core_meanings))
    nouns = [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')]
    verbs = [t.text for t in doc if t.pos_ == 'VERB']
    adjs  = [t.text for t in doc if t.pos_ == 'ADJ']

    possession = {'name','son','daughter','house','heart',
                  'brother','sister','father','mother','lord'}

    if verbs and nouns:
        return f'{verbs[0]} the {nouns[0]}'
    if len(nouns) >= 2:
        n1, n2 = nouns[0], nouns[1]
        return f'my {n1} is {n2}' if n1.lower() in possession else f'{n1} of {n2}'
    if nouns and adjs:
        return f'the {adjs[0]} {nouns[0]}'
    if adjs:
        return f'it is {adjs[0]}'
    return ' '.join(core_meanings)


def transliterate(gardiner_codes):
    codes = [c.lower().strip() for c in gardiner_codes]
    phonetics, glyphs, sign_meanings, unknown = [], [], [], []

    for code in codes:
        if code in GARDINER_MAP:
            info = GARDINER_MAP[code]
            phonetics.append(info['phonetic'])
            glyphs.append(info['unicode'])
            sign_meanings.append(info['meaning'])
        else:
            phonetics.append('?')
            glyphs.append('□')
            sign_meanings.append('unknown')
            unknown.append(code)

    ph_clean = [p for p in phonetics if p and p != '?']
    full     = ''.join(ph_clean)

    token_results, core_meanings = [], []
    i = 0
    while i < len(ph_clean):
        matched = False
        for length in range(min(4, len(ph_clean) - i), 0, -1):
            combined = ''.join(ph_clean[i:i+length])
            if combined in EGYPTIAN_DICT:
                ml   = EGYPTIAN_DICT[combined]
                core = extract_core_meaning(ml)
                token_results.append({'phonetic': combined, 'meaning': ' | '.join(ml), 'core': core, 'found': True})
                core_meanings.append(core)
                i += length
                matched = True
                break
        if not matched:
            token_results.append({'phonetic': ph_clean[i], 'meaning': f'[{ph_clean[i]}]', 'core': ph_clean[i], 'found': False})
            i += 1

    candidates = [full] if full else []
    for size in range(len(ph_clean), 0, -1):
        for start in range(len(ph_clean) - size + 1):
            w = ''.join(ph_clean[start:start+size])
            if w and w not in candidates:
                candidates.append(w)

    found_words = []
    for c in candidates:
        if c in EGYPTIAN_DICT:
            ml = EGYPTIAN_DICT[c]
            found_words.append({
                'transliteration': c,
                'meaning':         ' | '.join(ml),
                'confidence':      'high' if c == full else 'partial',
            })

    high = [w for w in found_words if w['confidence'] == 'high']
    best = high[0] if high else (found_words[0] if found_words else None)

    return {
        'input_codes'  : codes,
        'glyphs'       : glyphs,
        'glyph_str'    : ' '.join(glyphs),
        'per_sign'     : list(zip(codes, phonetics, sign_meanings)),
        'phonetics'    : phonetics,
        'phonetic_str' : ' '.join(ph_clean),
        'assembled'    : full,
        'token_results': token_results,
        'found_words'  : found_words,
        'unknown_codes': unknown,
        'best_meaning' : best['meaning'] if best else None,
        'sentence'     : build_sentence_spacy(core_meanings),
        'core_meanings': core_meanings,
    }


# ─────────────────────────────────────────────────────────────
# TRANSLATION — Seed-X-PPO-7B
# ─────────────────────────────────────────────────────────────

def translate_to_arabic(english_text):
    """
    ترجمة إنجليزي → عربي بـ Seed-X-PPO-7B
    الـ prompt الصح حسب الـ docs:
      "Translate the following English sentence into Arabic:\n{text} <ar>"
    """
    if not english_text or english_text.startswith('['):
        return ''

    prompt = f'Translate the following English sentence into Arabic:\n{english_text} <ar>'

    try:
        inputs = seed_x_tokenizer(prompt, return_tensors='pt')
        inputs.pop('token_type_ids', None)
        inputs = {k: v.to(seed_x_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seed_x_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=seed_x_tokenizer.eos_token_id,
            )
        arabic = seed_x_tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip()
        print(f'🔄 Translated: "{english_text}" → "{arabic}"')
        return arabic if arabic else english_text
    except Exception as e:
        print(f'❌ Translation error: {str(e)[:100]}')
        return english_text
# ─────────────────────────────────────────────────────────────

def analyze_sentiment(text):
    if not text or text.startswith('['):
        return 'neutral', 0.5
    try:
        inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            scores = torch.softmax(sent_model(**inputs).logits, dim=1).numpy()[0]
        idx    = int(np.argmax(scores))
        labels = ['negative', 'neutral', 'positive']
        return labels[idx], round(float(scores[idx]), 3)
    except:
        return 'neutral', 0.5


# ─────────────────────────────────────────────────────────────
# INTENTION
# ─────────────────────────────────────────────────────────────

def detect_intention(text, phonetics):
    combined = (text + ' ' + phonetics).lower()
    words    = set(combined.split())
    scores   = {i: len(words & d['keywords']) for i, d in INTENTION_MAP.items() if words & d['keywords']}
    if not scores:
        return 'descriptive', 'وصفي'
    best = max(scores, key=scores.get)
    return best, INTENTION_MAP[best]['arabic']


# ─────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────

def run_nlp_pipeline(gardiner_codes, verbose=True):
    # Step 1: Transliteration
    trans        = transliterate(gardiner_codes)
    best_meaning = trans['best_meaning']
    sentence     = trans['sentence']

    if sentence:
        english, method = sentence, 'spacy-nlp'
    elif best_meaning:
        english, method = best_meaning, 'dictionary'
    else:
        english, method = f'[unknown: {trans["assembled"]}]', 'none'

    # Step 2: Translation → Arabic (Seed-X-PPO-7B)
    arabic = translate_to_arabic(english)

    # Step 3: Sentiment
    sentiment_text        = (sentence + ' ' + (best_meaning or '')).strip()
    sentiment, sent_score = analyze_sentiment(sentiment_text)

    # Step 4: Intention
    intention_en, intention_ar = detect_intention(sentiment_text, trans['phonetic_str'])

    result = {
        'input'        : gardiner_codes,
        'glyphs'       : trans['glyph_str'],
        'per_sign'     : [list(x) for x in trans['per_sign']],
        'phonetics'    : trans['phonetic_str'],
        'assembled'    : trans['assembled'],
        'token_results': trans['token_results'],
        'found_words'  : trans['found_words'][:3],
        'sentence'     : sentence,
        'english'      : english,
        'trans_method' : method,
        'arabic'        : arabic,
        'sentiment'    : sentiment,
        'sent_score'   : sent_score,
        'intention_en' : intention_en,
        'intention_ar' : intention_ar,
    }

    if verbose:
        emoji = '😊' if sentiment=='positive' else '😞' if sentiment=='negative' else '😐'
        print('=' * 58)
        print(f'  🏺  HIEROGLYPH NLP PIPELINE')
        print('=' * 58)
        print(f'  Input    : {gardiner_codes}')
        print(f'  Glyphs   : {result["glyphs"]}')
        print(f'  Phonetics: {result["phonetics"]}')
        print(f'  Sentence : {sentence}')
        print(f'  English  : {english} [{method}]')
        print(f'  Arabic   : {arabic}')
        print(f'  Sentiment: {emoji} {sentiment} ({sent_score})')
        print(f'  Intention: {intention_en} — {intention_ar}')
        print('=' * 58)

    return result


print('✅ All NLP functions ready')

✅ All NLP functions ready


## Cell 8 — Quick Test

In [9]:
# تست سريع — المفروض Arabic يطلع عربي مش إنجليزي
print('Test 1: name sun')
r1 = run_nlp_pipeline(['d21', 'n35', 'n5'], verbose=True)
print()

print('Test 2: house')
r2 = run_nlp_pipeline(['o1'], verbose=True)
print()

print('Test 3: offering the king')
r3 = run_nlp_pipeline(['r4', 'x8', 'a42'], verbose=True)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Test 1: name sun


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔄 Translated: "my name is sun" → "اسمي سون."
  🏺  HIEROGLYPH NLP PIPELINE
  Input    : ['d21', 'n35', 'n5']
  Glyphs   : 𓂋 𓈖 𓇳
  Phonetics: r n ra
  Sentence : my name is sun
  English  : my name is sun [spacy-nlp]
  Arabic   : اسمي سون.
  Sentiment: 😐 neutral (0.801)
  Intention: rebirth — التجدد والبعث

Test 2: house


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔄 Translated: "house" → "منزل."
  🏺  HIEROGLYPH NLP PIPELINE
  Input    : ['o1']
  Glyphs   : 𓉐
  Phonetics: pr
  Sentence : house
  English  : house [spacy-nlp]
  Arabic   : منزل.
  Sentiment: 😐 neutral (0.655)
  Intention: marriage — الزواج

Test 3: offering the king
🔄 Translated: "offering give" → "عرض منحة."
  🏺  HIEROGLYPH NLP PIPELINE
  Input    : ['r4', 'x8', 'a42']
  Glyphs   : 𓊵 𓏕 𓀯
  Phonetics: Htp di
  Sentence : offering give
  English  : offering give [spacy-nlp]
  Arabic   : عرض منحة.
  Sentiment: 😊 positive (0.667)
  Intention: offering — تقديم قربان


## Cell 9 — Flask API + Ngrok Tunnel

> ⚠️ **مهم جداً:**
> - روح `dashboard.ngrok.com/get-started/your-authtoken`
> - انسخ الـ **Authtoken** الكامل (مش الـ ID)
> - الـ token الحقيقي بيبدأ بـ `2` وبيكون طويل (~50 حرف)
> - **مش** `cr_3BBILR8...` ده الـ ID مش التوكن

In [11]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
import threading

# ════════════════════════════════════════════════════════
# ⚠️  ضع الـ Authtoken الكامل هنا
#     من: dashboard.ngrok.com/get-started/your-authtoken
#     الـ token الصح بيبدأ بـ أرقام وبيكون ~50 حرف
NGROK_TOKEN = "3BBILR8WLOgh6uQGi4jU0aN6YoR_68QjQ4J4eRxCyhsjHLWMr"
# ════════════════════════════════════════════════════════

conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)
CORS(app)


@app.route('/api/decipher', methods=['POST'])
def decipher():
    try:
        data  = request.get_json()
        codes = data.get('codes', [])
        if not codes:
            return jsonify({'error': 'Missing codes'}), 400
        result = run_nlp_pipeline(codes, verbose=False)
        return jsonify({'success': True, 'data': result}), 200
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500


@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({
        'status'      : 'ready',
        'model'       : 'Seed-X-PPO-7B',
        'signs_loaded': len(GARDINER_MAP),
        'dict_loaded' : len(EGYPTIAN_DICT),
    }), 200


@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify({
        'simple_word'   : {'codes': ['o1'],              'description': 'House'},
        'name_sun'      : {'codes': ['d21','n35','n5'],  'description': 'My name is sun'},
        'royal_offering': {'codes': ['r4','x8','a42'],   'description': 'Offering of the king'},
        'sun_god'       : {'codes': ['n5','r8','f35'],   'description': 'Sun god'},
        'son_of_ra'     : {'codes': ['o4','n5'],         'description': 'Son of Ra'},
    }), 200


def run_server():
    app.run(port=5000, use_reloader=False)


# ── شغّل السيرفر في thread منفصل ──────────────────────
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# ── افتح الـ tunnel ────────────────────────────────────
tunnel     = ngrok.connect(5000)
PUBLIC_URL = tunnel.public_url

print('═' * 62)
print('🔗  URL بتاعك — انسخه وحطه في main.js:')
print(f'\n    {PUBLIC_URL}\n')
print('═' * 62)
print('✅  السيرفر شغال — اتركه يرن')
print('⚠️  متغلقش الـ session دي على كاجل')

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


══════════════════════════════════════════════════════════════
🔗  URL بتاعك — انسخه وحطه في main.js:

    https://irretrievably-unsimpering-darrin.ngrok-free.dev

══════════════════════════════════════════════════════════════
✅  السيرفر شغال — اتركه يرن
⚠️  متغلقش الـ session دي على كاجل


127.0.0.1 - - [20/Mar/2026 03:40:02] "OPTIONS /api/decipher HTTP/1.1" 200 -
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
127.0.0.1 - - [20/Mar/2026 03:40:03] "POST /api/decipher HTTP/1.1" 200 -


🔄 Translated: "monument" → "نصب تذكاري."


...

127.0.0.1 - - [20/Mar/2026 03:53:59] "OPTIONS /api/decipher HTTP/1.1" 200 -
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
127.0.0.1 - - [20/Mar/2026 03:54:00] "POST /api/decipher HTTP/1.1" 200 -


🔄 Translated: "sun god ra" → "إله الشمس، را."


127.0.0.1 - - [20/Mar/2026 03:54:10] "OPTIONS /api/decipher HTTP/1.1" 200 -
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
127.0.0.1 - - [20/Mar/2026 03:54:11] "POST /api/decipher HTTP/1.1" 200 -


🔄 Translated: "sun of god" → "شمس الإله."


127.0.0.1 - - [20/Mar/2026 03:54:52] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:54:52] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:54:53] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:54:55] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:00] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:00] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:00] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:01] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:01] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:02] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:03] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:04] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:08] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [20/Mar/2026 03:55:09] "GET /api/e